# 04. Controls: additive baseline and random vectors

Question: does the protein input matter beyond identifying the protein?

Two controls:

1. **Additive baseline, no network.** Predict a test pair as (mean affinity of its drug + mean affinity of its protein - global mean), fitted on the training rows. It uses no protein sequence or structure at all, only protein identity.
2. **Random-vector control.** Every protein gets a fixed random 1024-dim vector (same sequence, same vector) and PLMNet is trained exactly like the real language-model arms: 5 folds, median best epoch, full-data model, same seed 42.

If the random vectors reach about the same MSE and CI as ESM C, ProtT5 and the structure arm, then on this split the protein branch only works as an identity signal.

## 1. Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Config

In [2]:
import os
from glob import glob

def find_folder(name):
    path = f"/content/drive/MyDrive/{name}"
    if not os.path.isdir(path):
        shared = glob(f"/content/drive/.shortcut-targets-by-id/*/{name}")
        if shared:
            path = shared[0]
    assert os.path.isdir(path), f"Folder not found: {path}"
    return path

DRIVE_ROOT = find_folder("Diplomska_ctrl")
V2_ROOT = find_folder("Diplomska_v2")
PDB_SET = "kinase_domain"
PLM_MODEL = "random_1024"
STOP_AFTER_FOLDS = None

os.environ["DTA_DATA_ROOT"] = DRIVE_ROOT
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
print("control folder:", DRIVE_ROOT)
print("v2 folder (read only):", V2_ROOT)
print("DTA_DATA_ROOT:", os.environ["DTA_DATA_ROOT"])

control folder: /content/drive/MyDrive/Diplomska_ctrl
v2 folder (read only): /content/drive/MyDrive/Diplomska_v2
DTA_DATA_ROOT: /content/drive/MyDrive/Diplomska_ctrl


## 3. GPU

In [ ]:
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
!nvidia-smi --query-gpu=name,memory.total --format=csv

Torch version: 2.11.0+cu130
CUDA available: True
name, memory.total [MiB]
Tesla T4, 15360 MiB


## 4. Install packages

In [3]:
!pip install -q torch_geometric biopython

## 5. Go to the folder

In [4]:
%cd {DRIVE_ROOT}
!pwd

/content/drive/MyDrive/Diplomska_ctrl
/content/drive/MyDrive/Diplomska_ctrl


## 6. Check the uploaded code

In [17]:
import argparse, json
import numpy as np
import pandas as pd

required = [
    "training_5_folds.py", "testing.py", "report_best_epochs.py", "compute_protein_embeddings.py",
    "src/__init__.py", "src/common/__init__.py", "src/structure/__init__.py", "src/sequence/__init__.py",
    "src/common/seeding.py", "src/common/paths.py", "src/common/run_tag.py", "src/common/structure_sets.py",
    "src/common/conditions.py", "src/common/checkpointing.py", "src/common/dta_dataset.py",
    "src/common/emetrics.py", "src/common/creating_train_and_test_set.py", "src/common/drug_representation.py",
    "src/structure/gnn_net.py", "src/structure/protein_representation.py",
    "src/sequence/plm_net.py", "src/sequence/protein_embeddings.py",
]
missing = [f for f in required if not os.path.exists(f)]
print("Missing code files:", missing or "none")

from src.common.run_tag import add_run_args, run_tag
from src.common.seeding import SEED
parser = argparse.ArgumentParser()
add_run_args(parser)
help_text = parser.format_help()
print("--pdb-set accepts", PDB_SET + ":", PDB_SET in help_text)
print("--plm-model accepts random_1024:", "random_1024" in help_text)

from src.common.paths import paths_for
PATHS = paths_for("davis", PDB_SET)
assert PATHS.root == DRIVE_ROOT, f"Data root {PATHS.root} is not {DRIVE_ROOT}"
print("data root:", PATHS.root)

TAG_RANDOM = run_tag("plm", PLM_MODEL, SEED, False, PDB_SET)
print("Run tag:", TAG_RANDOM, "(expected random_1024_pdb2)")
assert TAG_RANDOM == "random_1024_pdb2"

Missing code files: none
--pdb-set accepts kinase_domain: True
--plm-model accepts random_1024: True
data root: /content/drive/MyDrive/Diplomska_ctrl
Run tag: random_1024_pdb2 (expected random_1024_pdb2)


## 7. Copy data and reference results from main folder
Copied: the kinase-domain data (PDB files, contact maps, protein graphs, ligand graphs, folds, pair table).

In [12]:
import shutil

SRC_DAVIS = os.path.join(V2_ROOT, "data", "davis")
DST_DAVIS = os.path.join(DRIVE_ROOT, "data", "davis")
assert os.path.isdir(SRC_DAVIS), SRC_DAVIS

SKIP_TOP_LEVEL = {"pdb", "contact_maps", "protein_graphs", "protein_embeddings", "protein_embeddings_kinase_domain"}

def ignore(dirpath, names):
    if os.path.abspath(dirpath) == os.path.abspath(SRC_DAVIS):
        return SKIP_TOP_LEVEL & set(names)
    return set()

def copy_if_new(src, dst):
    if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
        return dst
    return shutil.copy2(src, dst)

shutil.copytree(SRC_DAVIS, DST_DAVIS, ignore=ignore, copy_function=copy_if_new, dirs_exist_ok=True)

REFERENCE_DIR = os.path.join(DRIVE_ROOT, "reference_v2")
os.makedirs(REFERENCE_DIR, exist_ok=True)
os.makedirs(PATHS.models, exist_ok=True)
os.makedirs(PATHS.results, exist_ok=True)
REFERENCE_FILES = [
    "result_PLMNet_davis_esmc_300m_pdb2.txt", "result_PLMNet_davis_esmc_300m_pdb2_full.txt",
    "result_PLMNet_davis_esmc_600m_pdb2.txt", "result_PLMNet_davis_esmc_600m_pdb2_full.txt",
    "result_PLMNet_davis_prot_t5_xl_uniref50_pdb2.txt", "result_PLMNet_davis_prot_t5_xl_uniref50_pdb2_full.txt",
    "result_GNNNet_davis_pdb2.txt", "result_GNNNet_davis_pdb2_full.txt",
]
for name in REFERENCE_FILES:
    src = os.path.join(V2_ROOT, "results_davis", name)
    if os.path.exists(src):
        copy_if_new(src, os.path.join(REFERENCE_DIR, name))
    else:
        print("reference not found in v2:", name)
print("reference results:", sorted(os.listdir(REFERENCE_DIR)))

reference results: ['result_GNNNet_davis_pdb2.txt', 'result_GNNNet_davis_pdb2_full.txt', 'result_PLMNet_davis_esmc_300m_pdb2.txt', 'result_PLMNet_davis_esmc_300m_pdb2_full.txt', 'result_PLMNet_davis_esmc_600m_pdb2.txt', 'result_PLMNet_davis_esmc_600m_pdb2_full.txt', 'result_PLMNet_davis_prot_t5_xl_uniref50_pdb2.txt', 'result_PLMNet_davis_prot_t5_xl_uniref50_pdb2_full.txt']


## 8. Check the copied data

In [ ]:
names = pd.read_csv(PATHS.csv)["protein"].unique()
pdb_missing = [n for n in names if not os.path.exists(PATHS.pdb_file(n))]
print(f"proteins in pair table: {len(names)} (expected 442)")
print(f"PDB files missing for {len(pdb_missing)} proteins <- expected 0")
print(f"protein graphs: {len(glob(os.path.join(PATHS.protein_graph_gml, '*_contact_map_graph.gml')))} (expected 442)")
print(f"ligand graphs: {len(glob(os.path.join(PATHS.ligand_graphs, '*.gml')))} (expected 68)")
with open(PATHS.train_folds) as f:
    print("fold sizes:", [len(x) for x in json.load(f)], "(expected 5010, 5009, 5009, 5009, 5009)")

proteins in pair table: 442 (expected 442)
PDB files missing for 0 proteins <- expected 0
protein graphs: 442 (expected 442)
ligand graphs: 68 (expected 68)
fold sizes: [5010, 5009, 5009, 5009, 5009] (expected 5010, 5009, 5009, 5009, 5009)


## 9. Helpers

In [6]:
import re, statistics, subprocess

QUIET_PREFIXES = ("Train epoch:", "Training on ", "Predicting for validation data",
                  "Make prediction for", "MSE improved at epoch", "No improvement since epoch")
VALIDATION_LINE = re.compile(r"^Epoch (\d+) - Validation MSE")

def run(args, stop_marker=None, every=10):
    cmd = ["python", "-u"] + args
    print("Running:", " ".join(cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    stopped = False
    try:
        for line in proc.stdout:
            match = VALIDATION_LINE.match(line)
            if match:
                if int(match.group(1)) % every == 0:
                    print(line, end="")
                continue
            if line.startswith(QUIET_PREFIXES):
                continue
            print(line, end="")
            if stop_marker and stop_marker in line:
                stopped = True
                print("\nStopping here as requested. Rerun this cell to continue; finished folds are skipped.")
                break
    finally:
        if proc.poll() is None:
            proc.terminate()
        proc.wait()
    if not stopped and proc.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {proc.returncode}, see the output above.")

FLAGS_RANDOM = ["--protein-repr", "plm", "--pdb-set", PDB_SET, "--plm-model", PLM_MODEL]

def stop_marker():
    return f"Training for fold {STOP_AFTER_FOLDS}..." if STOP_AFTER_FOLDS else None

def median_best_epoch(tag):
    path = os.path.join(PATHS.results, f"best_epochs_{tag}.json")
    with open(path) as f:
        records = json.load(f)
    folds = [str(i) for i in range(5)]
    absent = [i for i in folds if i not in records]
    assert not absent, f"Folds {absent} are not finished yet in {path}"
    print(f"{'fold':>4} {'best_epoch':>10} {'last_epoch':>10} {'best_val_mse':>13}")
    for i in folds:
        r = records[i]
        print(f"{i:>4} {r['best_epoch']:>10} {r['last_epoch']:>10} {r['best_val_mse']:>13.4f}")
    median = int(round(statistics.median(records[i]["best_epoch"] for i in folds)))
    print("Median best epoch (use for the full-data model):", median)
    return median

def show_result(model_name, tag, full=False):
    path = os.path.join(PATHS.results, f"result_{model_name}_davis_{tag}{'_full' if full else ''}.txt")
    print(path)
    with open(path) as f:
        print(f.read())

def read_mean(path, metric):
    if not os.path.exists(path):
        return None
    with open(path) as f:
        for line in f:
            if line.startswith(metric + " ") and "mean:" in line:
                return float(line.split("mean:")[1].split()[0])
    return None

# Part 1: additive baseline (no protein sequence, no structure)

In [9]:
df = pd.read_csv(PATHS.csv)
print("columns:", list(df.columns))
print(df.head(3).to_string())
print("rows:", len(df), "(expected 30056)")

def pick(cands, exact=()):
    for c in df.columns:
        if c.lower() in exact:
            return c
    for c in df.columns:
        if any(k in c.lower() for k in cands):
            return c
    raise RuntimeError(f"No column found among {list(df.columns)}")

DRUG_COL = pick(("smiles", "drug", "ligand", "compound"))
AFF_COL = pick(("affin",), exact=("y", "label", "pkd", "kd"))
PROT_COL = "protein"
print("drug column:", DRUG_COL, "| affinity column:", AFF_COL, "| protein column:", PROT_COL)
print("unique drugs:", df[DRUG_COL].nunique(), "(expected 68) | unique proteins:", df[PROT_COL].nunique(), "(expected 442)")

def read_indices(path):
    with open(path) as f:
        text = f.read()
    return json.loads(text)

folds = read_indices(PATHS.train_folds)
train_idx = [int(i) for fold in folds for i in fold]
test_idx = [int(i) for i in read_indices(PATHS.test_fold)]
print("train rows:", len(train_idx), "(expected 25046) | test rows:", len(test_idx), "(expected 5010)")
print("max index:", max(train_idx + test_idx), "| rows in table:", len(df))
assert max(train_idx + test_idx) < len(df), "Indices do not point into the pair table"
assert not (set(train_idx) & set(test_idx)), "Train and test overlap"
assert df[AFF_COL].notna().all(), "NaN affinities in the table"
print("affinity range:", df[AFF_COL].min(), "to", df[AFF_COL].max())

columns: ['ligand', 'protein', 'label']
                                                ligand      protein     label
0  CC1=C2C=C(C=CC2=NN1)C3=CC(=CN=C3)OCC(CC4=CC=CC=C4)N         AAK1  7.366532
1  CC1=C2C=C(C=CC2=NN1)C3=CC(=CN=C3)OCC(CC4=CC=CC=C4)N  ABL1(E255K)  5.000000
2  CC1=C2C=C(C=CC2=NN1)C3=CC(=CN=C3)OCC(CC4=CC=CC=C4)N  ABL1(F317I)  5.000000
rows: 30056 (expected 30056)
drug column: ligand | affinity column: label | protein column: protein
unique drugs: 68 (expected 68) | unique proteins: 442 (expected 442)
train rows: 25046 (expected 25046) | test rows: 5010 (expected 5010)
max index: 30055 | rows in table: 30056
affinity range: 5.0 to 10.795880017344077


In [16]:
def concordance(y, p, chunk=500):
    y = np.asarray(y, dtype=np.float64); p = np.asarray(p, dtype=np.float64)
    num = 0.0; den = 0
    for i in range(0, len(y), chunk):
        dy = y[i:i + chunk, None] - y[None, :]
        dp = p[i:i + chunk, None] - p[None, :]
        pairs = dy > 0
        num += float((pairs & (dp > 0)).sum()) + 0.5 * float((pairs & (dp == 0)).sum())
        den += int(pairs.sum())
    return num / den

train = df.iloc[train_idx]
test = df.iloc[test_idx]
y_train = train[AFF_COL].to_numpy(); y_test = test[AFF_COL].to_numpy()
mu = y_train.mean()

drug_mean = train.groupby(DRUG_COL)[AFF_COL].mean()
prot_mean = train.groupby(PROT_COL)[AFF_COL].mean()

def additive_fit(train, mu, iters=200):
    a = pd.Series(0.0, index=train[DRUG_COL].unique())
    b = pd.Series(0.0, index=train[PROT_COL].unique())
    for _ in range(iters):
        r = train[AFF_COL] - mu - train[PROT_COL].map(b)
        a = r.groupby(train[DRUG_COL]).mean()
        r = train[AFF_COL] - mu - train[DRUG_COL].map(a)
        b = r.groupby(train[PROT_COL]).mean()
    return a, b

a, b = additive_fit(train, mu)

predictions = {
    "global mean": np.full(len(test), mu),
    "drug mean only": test[DRUG_COL].map(drug_mean).fillna(mu).to_numpy(),
    "protein mean only": test[PROT_COL].map(prot_mean).fillna(mu).to_numpy(),
    "drug mean + protein mean - global": (test[DRUG_COL].map(drug_mean).fillna(mu)
                                          + test[PROT_COL].map(prot_mean).fillna(mu) - mu).to_numpy(),
    "additive fit (drug + protein effects)": (mu + test[DRUG_COL].map(a).fillna(0.0)
                                              + test[PROT_COL].map(b).fillna(0.0)).to_numpy(),
}
unseen_prot = int((~test[PROT_COL].isin(train[PROT_COL])).sum())
unseen_drug = int((~test[DRUG_COL].isin(train[DRUG_COL])).sum())
print(f"test rows with a protein / drug not seen in training: {unseen_prot} / {unseen_drug}")
print()
print(f"{'predictor':40} {'MSE':>8} {'CI':>8}")
BASELINE = {}
for name, pred in predictions.items():
    mse = float(np.mean((y_test - pred) ** 2)); ci = concordance(y_test, pred)
    BASELINE[name] = (mse, ci)
    print(f"{name:40} {mse:8.4f} {ci:8.4f}")
print()
print("Reference, full-data models (seed 42): ESM C 300M 0.1625 / 0.9116.")

test rows with a protein / drug not seen in training: 0 / 0

predictor                                     MSE       CI
global mean                                0.8015   0.5000
drug mean only                             0.6453   0.7435
protein mean only                          0.7264   0.6554
drug mean + protein mean - global          0.5594   0.7911
additive fit (drug + protein effects)      0.5583   0.7919

Reference, full-data models (seed 42): ESM C 300M 0.1625 / 0.9116.


# Part 2: random embeddings

In [ ]:
run(["compute_protein_embeddings.py", "--model", PLM_MODEL, "--pdb-set", PDB_SET])

Running: python -u compute_protein_embeddings.py --model random_1024 --pdb-set kinase_domain
Structures: kinase_domain (/content/drive/MyDrive/Diplomska_ctrl/data/davis/pdb_kinase_domain)
442 proteins, 428 unique sequences, 428 to embed with random_1024
Loading random_1024 on cuda
[1/428] embedded 378 residues
[2/428] embedded 378 residues
[3/428] embedded 262 residues
[4/428] embedded 264 residues
[5/428] embedded 268 residues
[6/428] embedded 269 residues
[7/428] embedded 315 residues
[8/428] embedded 315 residues
[9/428] embedded 190 residues
[10/428] embedded 259 residues
[11/428] embedded 262 residues
[12/428] embedded 258 residues
[13/428] embedded 294 residues
[14/428] embedded 269 residues
[15/428] embedded 274 residues
[16/428] embedded 255 residues
[17/428] embedded 252 residues
[18/428] embedded 252 residues
[19/428] embedded 258 residues
[20/428] embedded 258 residues
[21/428] embedded 417 residues
[22/428] embedded 412 residues
[23/428] embedded 260 residues
[24/428] embed

In [ ]:
cache = os.path.join(PATHS.protein_embeddings, PLM_MODEL)
print("cache folder:", cache)
print("files:", len(os.listdir(cache)), "| pooled vectors:", len(glob(os.path.join(cache, "*.pooled.pt"))),
      "(expected 428 pooled, plus one index file)")

cache folder: /content/drive/MyDrive/Diplomska_ctrl/data/davis/protein_embeddings_kinase_domain/random_1024
files: 429 | pooled vectors: 428 (expected 428 pooled, plus one index file)


# Part 3: PLMNet on random vectors

## 3.1 Five folds

In [ ]:
run(["training_5_folds.py"] + FLAGS_RANDOM, stop_marker=stop_marker())

Running: python -u training_5_folds.py --protein-repr plm --pdb-set kinase_domain --plm-model random_1024
Dataset:  davis
Mode:  cv
Protein representation:  plm
Structure set:  kinase_domain
Protein language model:  random_1024
Standardized embeddings:  False
Seed:  42
Learning rate:  0.001
Epochs:  2000
Early stop patience:  100
Data root:  /content/drive/MyDrive/Diplomska_ctrl
Using device: cuda
Run tag:  random_1024_pdb2
Loaded random_1024 embeddings for 442 proteins, dim 1024
Training for fold 0...
/content/drive/MyDrive/Diplomska_ctrl/training_5_folds.py:39: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  return DataLoader(
/content/drive/MyDrive/Diplomska_ctrl/training_5_folds.py:111: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  valid_loader = DataLoader(
Epoch 10 - Validation MSE: 0.7356505393981934, Best MSE: 0.61538165807724
Epoch 20 - Validation MSE: 0.641675591468811, Best MSE: 0.5518674254417419
Epoch 30 - V

## 3.2 Median best epoch

In [ ]:
FULL_EPOCHS = median_best_epoch(TAG_RANDOM)

fold best_epoch last_epoch  best_val_mse
   0        617        717        0.2453
   1        858        958        0.2223
   2        516        616        0.2304
   3        540        640        0.2480
   4        674        774        0.2227
Median best epoch (use for the full-data model): 617


## 3.3 Test the five fold models

In [ ]:
run(["testing.py"] + FLAGS_RANDOM)
show_result("PLMNet", TAG_RANDOM)

Running: python -u testing.py --protein-repr plm --pdb-set kinase_domain --plm-model random_1024
Loaded random_1024 embeddings for 442 proteins, dim 1024
dataset: davis
device: cuda:0
seed: 42
structure set: kinase_domain
Fold 0 metrics:
  rmse: 0.5042993653322364
  mse: 0.25431784987449646
  pearson: 0.826736461431165
  spearman: 0.6753078919779558
  ci: 0.8755518281602706
  cindex: 0.877456718201322
  rm2: 0.6611340641975403
/content/drive/MyDrive/Diplomska_ctrl/testing.py:89: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend(loc=0, numpoints=1)
Fold 1 metrics:
  rmse: 0.49223048915527423
  mse: 0.24229085445404053
  pearson: 0.8375166206371505
  spearman: 0.6750021702801755
  ci: 0.8752609912683077
  cindex: 0.876353386207444
  rm2: 0.6930707097053528
Fold 2 metrics:
  rmse: 0.5085873194696706
  mse: 0.25866106152534485
  pearson: 0.8245109705870312
  

## 3.4 Full-data model and its test score

In [7]:
# The full-data model was trained earlier with: training_5_folds.py --mode full --epochs <median best epoch>
run(["testing.py"] + FLAGS_RANDOM + ["--full"])
show_result("PLMNet", TAG_RANDOM, full=True)

Running: python -u testing.py --protein-repr plm --pdb-set kinase_domain --plm-model random_1024 --full
Loaded random_1024 embeddings for 442 proteins, dim 1024
dataset: davis
device: cuda:0
seed: 42
structure set: kinase_domain
Fold full metrics:
  rmse: 0.45284293702307726
  mse: 0.2050667256116867
  pearson: 0.8627210169502273
  spearman: 0.70231719739678
  ci: 0.8937208636645785
  cindex: 0.8937059456189502
  rm2: 0.7351962327957153
/content/drive/MyDrive/Diplomska_ctrl/testing.py:89: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend(loc=0, numpoints=1)

davis  folds evaluated: ['full']
ci (all pairs) is the value to compare with published numbers; cindex only counts pairs whose larger label has the larger row index.

fold full  rmse: 0.4528  mse: 0.2051  pearson: 0.8627  spearman: 0.7023  ci: 0.8937  cindex: 0.8937  rm2: 0.7352

rmse      mean: 0.452

## 3.5 Seeds 43 and 44 of the full-data model

In [14]:
SEEDS = [43, 44]

In [ ]:
def seed_tag(seed):
    tag = run_tag("plm", PLM_MODEL, seed, False, PDB_SET)
    assert tag.endswith(f"s{seed}"), tag
    return tag

def train_and_test_full(seed, epochs):
    tag = seed_tag(seed)
    flags = FLAGS_RANDOM + ["--seed", str(seed)]
    model_file = os.path.join(PATHS.models, f"model_PLMNet_davis_{tag}_full.model")
    result_file = os.path.join(PATHS.results, f"result_PLMNet_davis_{tag}_full.txt")
    print(f"=== seed {seed}, tag {tag}, {epochs} epochs ===")
    if os.path.exists(model_file):
        print("Model already finished, skipping training:", model_file)
    else:
        run(["training_5_folds.py"] + flags + ["--mode", "full", "--epochs", str(epochs)], every=50)
    if os.path.exists(result_file):
        print("Result already exists, skipping test:", result_file)
    else:
        run(["testing.py"] + flags + ["--full"])
    show_result("PLMNet", tag, full=True)

for seed in SEEDS:
    train_and_test_full(seed, FULL_EPOCHS)

=== seed 43, tag random_1024_pdb2_s43, 617 epochs ===
Running: python -u training_5_folds.py --protein-repr plm --pdb-set kinase_domain --plm-model random_1024 --seed 43 --mode full --epochs 617
Dataset:  davis
Mode:  full
Protein representation:  plm
Structure set:  kinase_domain
Protein language model:  random_1024
Standardized embeddings:  False
Seed:  43
Learning rate:  0.001
Epochs:  617
Data root:  /content/drive/MyDrive/Diplomska_ctrl
Using device: cuda
Run tag:  random_1024_pdb2_s43
Loaded random_1024 embeddings for 442 proteins, dim 1024
/content/drive/MyDrive/Diplomska_ctrl/training_5_folds.py:39: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  return DataLoader(
Saved full-data model after 617 epochs to /content/drive/MyDrive/Diplomska_ctrl/models_davis/model_PLMNet_davis_random_1024_pdb2_s43_full.model
Running: python -u testing.py --protein-repr plm --pdb-set kinase_domain --plm-model random_1024 --seed 43 --full
Loaded random_1024 embedding

# Summary

In [15]:
def full_path(model, tag):
    return os.path.join(PATHS.results, f"result_{model}_davis_{tag}_full.txt")

def ref(name):
    return os.path.join(REFERENCE_DIR, name)

rows = []
rows.append(("additive fit, no network", BASELINE["additive fit (drug + protein effects)"][0], BASELINE["additive fit (drug + protein effects)"][1], None, None))
rows.append(("random 1024 vectors (control)",
             read_mean(os.path.join(PATHS.results, f"result_PLMNet_davis_{TAG_RANDOM}.txt"), "mse"),
             read_mean(os.path.join(PATHS.results, f"result_PLMNet_davis_{TAG_RANDOM}.txt"), "ci"),
             read_mean(full_path("PLMNet", TAG_RANDOM), "mse"), read_mean(full_path("PLMNet", TAG_RANDOM), "ci")))
for label, model, tag in [("ESM C 300M", "PLMNet", "esmc_300m_pdb2"), ("ESM C 600M", "PLMNet", "esmc_600m_pdb2"),
                          ("ProtT5-XL", "PLMNet", "prot_t5_xl_uniref50_pdb2"), ("structure (GNNNet)", "GNNNet", "pdb2")]:
    five = ref(f"result_{model}_davis_{tag}.txt"); full = ref(f"result_{model}_davis_{tag}_full.txt")
    rows.append((label, read_mean(five, "mse"), read_mean(five, "ci"), read_mean(full, "mse"), read_mean(full, "ci")))

show = lambda v: f"{v:8.4f}" if v is not None else f"{'-':>8}"
print(f"{'arm':32} {'5-fold MSE':>10} {'5-fold CI':>10} {'full MSE':>10} {'full CI':>10}")
for label, m5, c5, mf, cf in rows:
    if label.startswith("additive"):
        print(f"{label:32} {'':>10} {'':>10} {show(m5):>10} {show(c5):>10}   (fitted on all training rows, scored on test)")
    else:
        print(f"{label:32} {show(m5):>10} {show(c5):>10} {show(mf):>10} {show(cf):>10}")

seed_vals = [(s, read_mean(full_path("PLMNet", run_tag("plm", PLM_MODEL, s, False, PDB_SET)), "mse"),
              read_mean(full_path("PLMNet", run_tag("plm", PLM_MODEL, s, False, PDB_SET)), "ci"))
             for s in [42] + SEEDS]
done = [(m, c) for _, m, c in seed_vals if m is not None]
if len(done) >= 2:
    print()
    print(f"random control, full data across {len(done)} seeds: MSE mean {statistics.mean(m for m, _ in done):.4f} "
          f"std {statistics.stdev(m for m, _ in done):.4f} | CI mean {statistics.mean(c for _, c in done):.4f} "
          f"std {statistics.stdev(c for _, c in done):.4f}")
print()

V2_RESULTS = os.path.join(V2_ROOT, "results_davis")
real_arms = [("ESM C 300M", "PLMNet", "plm", "esmc_300m"),
             ("ESM C 600M", "PLMNet", "plm", "esmc_600m"),
             ("ProtT5-XL", "PLMNet", "plm", "prot_t5_xl_uniref50"),
             ("structure (GNNNet)", "GNNNet", "structure", None)]

print("Real arms, full data, seeds 42/43/44 (read from the main folder)")
print(f"{'arm':22} {'MSE mean':>9} {'MSE std':>8} {'CI mean':>8} {'CI std':>7}")
for label, model, repr_, plm in real_arms:
    mses, cis = [], []
    for s in [42] + SEEDS:
        tag = run_tag(repr_, plm, s, False, PDB_SET)
        path = os.path.join(V2_RESULTS, f"result_{model}_davis_{tag}_full.txt")
        mses.append(read_mean(path, "mse")); cis.append(read_mean(path, "ci"))
    if None in mses or None in cis:
        print(f"{label:22} missing seed results")
        continue
    print(f"{label:22} {statistics.mean(mses):9.4f} {statistics.stdev(mses):8.4f} "
          f"{statistics.mean(cis):8.4f} {statistics.stdev(cis):7.4f}")

arm                              5-fold MSE  5-fold CI   full MSE    full CI
additive fit, no network                                   0.5583     0.7919   (fitted on all training rows, scored on test)
random 1024 vectors (control)        0.2525     0.8734     0.2051     0.8937
ESM C 300M                           0.1918     0.9001     0.1625     0.9116
ESM C 600M                           0.1848     0.9012     0.1586     0.9075
ProtT5-XL                            0.1959     0.8994     0.1666     0.9110
structure (GNNNet)                   0.2020     0.8993     0.1653     0.9104

random control, full data across 3 seeds: MSE mean 0.2096 std 0.0046 | CI mean 0.8915 std 0.0049

Real arms, full data, seeds 42/43/44 (read from the main folder)
arm                     MSE mean  MSE std  CI mean  CI std
ESM C 300M                0.1654   0.0050   0.9100  0.0031
ESM C 600M                0.1623   0.0047   0.9110  0.0030
ProtT5-XL                 0.1699   0.0079   0.9099  0.0009
structure (GN